# Notebook 01 - Consolidacion de datos fiscales

**Fuente:** Secretaria de Hacienda - https://www.argentina.gob.ar/economia/sechacienda/infoestadistica  
**Datos:** Sector Publico Base Caja (AIF) + Informe Mensual de Ingresos y Gastos (IMIG)  
**Cobertura:** enero 2020 - abril 2026 | 78 archivos Excel procesados

Este notebook lee los datasets consolidados desde GitHub y los exporta como un unico archivo Excel con multiples hojas, listo para usar en el **Notebook 02 de analisis**.

**Datasets:**
- `aif_consolidado.csv` — 27.964 registros AIF (Sector Publico Base Caja)
- `imig_consolidado.csv` — 7.950 registros IMIG (Ingresos y Gastos funcional)

**Unico mes sin datos AIF mensual:** Jun-2022 (Hacienda solo publico el acumulado I Semestre)

> **Como agregar nuevos meses:** copiar el Excel a `data/raw/` y correr `python src/consolidate.py`

In [ ]:
# Celda 1 - Instalacion de dependencias
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas openpyxl

import pandas as pd
import openpyxl
import os
print('OK - dependencias cargadas')

In [ ]:
# Celda 2 - URLs de los datos (rama main)
REPO_RAW = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'

AIF_URL  = f'{REPO_RAW}/output/aif_consolidado.csv'
IMIG_URL = f'{REPO_RAW}/output/imig_consolidado.csv'

print('Descargando datasets...')
df_aif  = pd.read_csv(AIF_URL,  parse_dates=['fecha'])
df_imig = pd.read_csv(IMIG_URL, parse_dates=['fecha'])

print(f'AIF : {len(df_aif):,} registros | {df_aif["fecha"].min().strftime("%Y-%m")} a {df_aif["fecha"].max().strftime("%Y-%m")}')
print(f'IMIG: {len(df_imig):,} registros')
df_aif.head(3)

In [ ]:
# Celda 3 - Resumen de cobertura
print('=== AIF - Conceptos disponibles ===')  
print(df_aif['concepto_codigo'].value_counts().to_string())
print()
print('=== AIF - Subsectores ===')
print(df_aif['subsector'].value_counts().to_string())
print()
print('=== Meses cubiertos por anio ===')
print(df_aif[df_aif['periodo']=='mensual'].groupby('anio')['mes'].nunique().to_string())

In [ ]:
# Celda 4 - Exportar a Excel consolidado
OUTPUT_FILE = 'datos_fiscales_consolidado.xlsx'

df_mensual = df_aif[df_aif['periodo'] == 'mensual']

# ── Helper: sector publico total (Adm.Nac + PAMI + Fondos) ───────────
def get_total(concepto):
    """Suma total_adm_nacional + pami_fdos_otros para cada mes."""
    nac  = df_mensual[df_mensual['concepto_codigo']==concepto].copy()
    nac  = nac[nac['subsector']=='total_adm_nacional'].set_index('fecha')['valor_millones_pesos']
    pami = df_mensual[df_mensual['concepto_codigo']==concepto].copy()
    pami = pami[pami['subsector']=='pami_fdos_otros'].set_index('fecha')['valor_millones_pesos']
    gen  = df_mensual[df_mensual['concepto_codigo']==concepto].copy()
    gen  = gen[gen['subsector']=='total_general'].set_index('fecha')['valor_millones_pesos']
    total = nac.add(pami.reindex(nac.index).fillna(0))
    if len(gen): total.update(gen)
    return total.reset_index().rename(columns={'valor_millones_pesos': concepto})

# ── Hoja: Resultado_pivot (Sector Publico Total) ──────────────────────
kpis = [
    'I_INGRESOS_CORRIENTES',
    'II_GASTOS_CORRIENTES',
    'II2_INTERESES',
    'II3_PRESTACIONES_SEG_SOCIAL',
    'XIV_RESULTADO_PRIMARIO',
    'XV_RESULTADO_FINANCIERO',
]
df_resultados = get_total(kpis[0])
for k in kpis[1:]:
    df_resultados = df_resultados.merge(get_total(k), on='fecha', how='outer')
df_resultados = df_resultados.sort_values('fecha').reset_index(drop=True)

# ── Hoja: Transferencias_provincias ──────────────────────────────────
df_prov = df_mensual[
    df_mensual['concepto_codigo'].isin([
        'II4b1_TRANSF_PROVINCIAS_CABA',
        'V2a_TRANSF_CAPITAL_PROVINCIAS',
    ])
].copy()
df_prov_pivot = df_prov.pivot_table(
    index=['fecha', 'concepto_codigo'],
    columns='subsector',
    values='valor_millones_pesos'
).reset_index()
df_prov_pivot.columns.name = None
df_prov_pivot['tipo'] = df_prov_pivot['concepto_codigo'].map({
    'II4b1_TRANSF_PROVINCIAS_CABA':  'Corrientes',
    'V2a_TRANSF_CAPITAL_PROVINCIAS': 'Capital',
})
cols_order = ['fecha', 'tipo'] + [c for c in df_prov_pivot.columns
                                   if c not in ('fecha', 'tipo', 'concepto_codigo')]
df_prov_pivot = df_prov_pivot[cols_order]

# ── Exportar ──────────────────────────────────────────────────────────
print(f'Exportando a {OUTPUT_FILE}...')
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_mensual.to_excel(writer, sheet_name='AIF_mensual',               index=False)
    df_aif[df_aif['periodo']=='acumulado'].to_excel(
                          writer, sheet_name='AIF_acumulado',            index=False)
    df_resultados.to_excel(writer, sheet_name='Resultado_pivot',         index=False)
    df_prov_pivot.to_excel(writer, sheet_name='Transferencias_provincias',index=False)
    df_imig.to_excel(      writer, sheet_name='IMIG',                    index=False)

print(f'Archivo generado: {OUTPUT_FILE}')
print()
print('Hojas exportadas:')
print('  AIF_mensual               - todos los conceptos x subsector, mensual')
print('  AIF_acumulado             - idem, acumulado')
print('  Resultado_pivot           - KPIs Sector Publico Total (Adm.Nac + PAMI)')
print('  Transferencias_provincias - corrientes Y capital a provincias, por subsector')
print('  IMIG                      - detalle funcional de ingresos y gastos')
print()
print(f'Cobertura: {df_aif["fecha"].min().strftime("%Y-%m")} a {df_aif["fecha"].max().strftime("%Y-%m")}')
print(f'Meses cubiertos (mensual): {df_mensual["fecha"].nunique()}')
print()
print('Resultado Primario anual - Sector Publico Total (MM ARS nominales):')
rp = df_resultados[['fecha','XIV_RESULTADO_PRIMARIO']].copy()
rp['anio'] = pd.to_datetime(rp['fecha']).dt.year
print(rp.groupby('anio')['XIV_RESULTADO_PRIMARIO'].sum().apply(lambda x: f'{x/1e6:.2f} B').to_string())

In [ ]:
# Celda 5 - Verificacion rapida de transferencias a provincias
print('Transferencias Tesoro -> Provincias/CABA (ultimos 12 meses, MM ARS):')
print()
for tipo, cod in [('Corrientes', 'II4b1_TRANSF_PROVINCIAS_CABA'), ('Capital', 'V2a_TRANSF_CAPITAL_PROVINCIAS')]:
    serie = df_mensual[
        (df_mensual['concepto_codigo'] == cod) &
        (df_mensual['subsector'] == 'tesoro_nacional')
    ].set_index('fecha')['valor_millones_pesos'].sort_index()
    print(f'  {tipo}:')
    print(serie.tail(12).to_string())
    print()

In [ ]:
# Celda 6 - Descargar el Excel (solo en Colab)
if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Descarga iniciada')
else:
    print(f'Archivo guardado en: {os.path.abspath(OUTPUT_FILE)}')